# 9.2 — Inspeção de Chunks

Análise dos chunks gerados pelo pipeline de chunking hierárquico parent-child.
Fonte: `data/processed/chunks.parquet`

In [ ]:
import sys
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

PROCESSED_DIR = ROOT / 'data' / 'processed'

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

chunks_path = PROCESSED_DIR / 'chunks.parquet'

try:
    df = pd.read_parquet(chunks_path)
    print(f'Chunks carregados: {len(df):,}')
    print(f'Colunas: {df.columns.tolist()}')
except FileNotFoundError:
    print(f'Arquivo nao encontrado: {chunks_path}')
    print('Execute src/04_chunk.py primeiro.')
    raise

In [ ]:
# Estatisticas gerais
print('=== Totais ===')
print(f'Total de chunks: {len(df):,}')
print()

print('=== Por chunk_type ===')
print(df['chunk_type'].value_counts().to_string())
print()

print('=== Tokens por chunk_type (mean / median / max) ===')
if 'tokens' in df.columns:
    stats = df.groupby('chunk_type')['tokens'].agg(['mean', 'median', 'max']).round(1)
    print(stats.to_string())
else:
    # Estima tokens por whitespace split se coluna nao existir
    df['tokens'] = df['texto'].fillna('').apply(lambda x: len(x.split()))
    print('(coluna tokens nao encontrada; estimativa por palavras)')
    stats = df.groupby('chunk_type')['tokens'].agg(['mean', 'median', 'max']).round(1)
    print(stats.to_string())

In [ ]:
# Histograma de tokens por tipo de chunk
tipos = df['chunk_type'].unique()
colors = {'child': 'steelblue', 'ementa': 'coral', 'fallback': 'mediumseagreen', 'table': 'mediumpurple'}

fig, axes = plt.subplots(1, len(tipos), figsize=(5 * len(tipos), 4), sharey=False)
if len(tipos) == 1:
    axes = [axes]

for ax, tipo in zip(axes, sorted(tipos)):
    subset = df[df['chunk_type'] == tipo]['tokens']
    ax.hist(subset, bins=40, color=colors.get(tipo, 'gray'), edgecolor='white')
    ax.axvline(subset.median(), color='red', linestyle='--', linewidth=1.2,
               label=f'Mediana: {subset.median():.0f}')
    ax.set_title(f'chunk_type = {tipo}\n(n={len(subset):,})')
    ax.set_xlabel('Tokens')
    ax.set_ylabel('Frequencia')
    ax.legend(fontsize=8)

plt.suptitle('Distribuicao de Tokens por Tipo de Chunk', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Exemplo de chunk de ementa
ementas = df[df['chunk_type'] == 'ementa']

if len(ementas) == 0:
    print('Nenhum chunk do tipo ementa encontrado.')
else:
    exemplo = ementas.sample(1, random_state=42).iloc[0]
    print('=== Exemplo de Chunk de Ementa ===')
    print(f'doc_id    : {exemplo.get("doc_id", "N/A")}')
    print(f'tipo_sigla: {exemplo.get("tipo_sigla", "N/A")}')
    print(f'numero    : {exemplo.get("numero", "N/A")}')
    print(f'ano       : {exemplo.get("ano", "N/A")}')
    print(f'tokens    : {exemplo.get("tokens", "N/A")}')
    print()
    print('--- TEXTO ---')
    print(str(exemplo['texto'])[:1500])

In [ ]:
# Hierarquia completa de 1 documento: ementa + texto_parent + children
# Pega um doc que tenha chunks child
children = df[df['chunk_type'] == 'child']

if len(children) == 0:
    print('Nenhum chunk child encontrado.')
else:
    # Escolhe doc com ao menos 3 children
    doc_counts = children['doc_id'].value_counts()
    docs_com_varios = doc_counts[doc_counts >= 3].index
    doc_alvo = docs_com_varios[0] if len(docs_com_varios) > 0 else children['doc_id'].iloc[0]

    doc_chunks = df[df['doc_id'] == doc_alvo]
    print(f'Documento: {doc_alvo}')
    print(f'Total de chunks neste doc: {len(doc_chunks)}')
    print(f'Tipos: {doc_chunks["chunk_type"].value_counts().to_dict()}')
    print()

    # Ementa
    em = doc_chunks[doc_chunks['chunk_type'] == 'ementa']
    if len(em) > 0:
        print('=== EMENTA ===')
        print(str(em.iloc[0]['texto'])[:600])
        print()

    # Primeiro artigo com children: texto_parent + children
    ch = doc_chunks[doc_chunks['chunk_type'] == 'child'].head(3)
    if len(ch) > 0:
        parent_texto = str(ch.iloc[0].get('texto_parent', '(texto_parent vazio)'))
        print('=== TEXTO PARENT (Art. completo) ===')
        print(parent_texto[:800])
        print()
        print('=== CHILDREN deste artigo ===')
        for i, row in ch.iterrows():
            print(f'  child parent_id={row.get("parent_id","?")} tokens={row.get("tokens","?")}')
            print(f'  {str(row["texto"])[:300]}')
            print()

In [ ]:
# Verificacao do prefixo contextual
# Todo chunk deve comecar com "{TIPO} {NUMERO}/{ANO}"
prefixo_re = re.compile(r'^(REN|REH|REA|DSP|PRT)\s+\d', re.IGNORECASE)

total = len(df)
com_prefixo = df['texto'].fillna('').apply(lambda x: bool(prefixo_re.match(str(x)))).sum()
pct = com_prefixo / total * 100

print(f'Chunks com prefixo contextual (tipo + numero): {int(com_prefixo):,} / {total:,} ({pct:.1f}%)')

# Por tipo de chunk
df['tem_prefixo'] = df['texto'].fillna('').apply(lambda x: bool(prefixo_re.match(str(x))))
print()
print('Por chunk_type:')
print(df.groupby('chunk_type')['tem_prefixo'].mean().mul(100).round(1).to_string())

In [ ]:
# Top 10 documentos com mais chunks
top_docs = df.groupby('doc_id').size().sort_values(ascending=False).head(10)
print('Top 10 documentos por numero de chunks:')
print(top_docs.to_string())

fig, ax = plt.subplots(figsize=(12, 4))
top_docs.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 10 Documentos com Mais Chunks')
ax.set_xlabel('Numero de Chunks')
plt.tight_layout()
plt.show()

In [ ]:
# Distribuicao de chunks por tipo_sigla
if 'tipo_sigla' in df.columns:
    pivot = df.groupby(['tipo_sigla', 'chunk_type']).size().unstack(fill_value=0)
    print('Chunks por tipo_sigla x chunk_type:')
    print(pivot.to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.plot(kind='bar', ax=ax, edgecolor='white')
    ax.set_title('Chunks por Tipo de Documento e Tipo de Chunk')
    ax.set_xlabel('Tipo Sigla')
    ax.set_ylabel('Quantidade')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='chunk_type', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()
else:
    print('Coluna tipo_sigla nao encontrada.')